# 📰 Fake News Detection using PyTorch

Build a Fake News Detection model using TF-IDF Vectorization and a Feedforward Neural Network in PyTorch.

# 📚 Import Required Libraries

In this section, we import all the necessary libraries for data manipulation, preprocessing, model building, training, and evaluation.

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, confusion_matrix



# 📂 Load the Dataset

Load the fake news dataset into a Pandas DataFrame and inspect its contents.

In [ ]:
ff = pd.read_csv("F_N_R_N.csv")
ff

,text,label
0,Gere faults Trump for blurring meaning of 'ref...,1
1,German parties start to find common ground in ...,1
2,Senate Democratic leader says Attorney General...,1
3,"Tennis: Kyrgios fined $10,000 for Shanghai wal...",1
4,Trump Threw Mar-A-Lago Fundraiser For Woman A...,0
...,...,...
45752,Heavily-Armed Man Claiming To Be Jesus Arrest...,0
45753,Evangelicals Across The Spectrum Are Clarifyin...,0
45754,"Senators question Kaleo' $4,500 tag on opioid ...",1
45755,CNN Does NOT Hold Back; Completely Makes Fun ...,0


# 🔍 Exploratory Data Analysis (EDA)

Analyze the dataset by checking:

- First few rows
- Last few rows
- Shape
- Summary statistics
- Data types
- Missing values
- Class distribution

In [ ]:
ff.head()

,text,label
0,Gere faults Trump for blurring meaning of 'ref...,1
1,German parties start to find common ground in ...,1
2,Senate Democratic leader says Attorney General...,1
3,"Tennis: Kyrgios fined $10,000 for Shanghai wal...",1
4,Trump Threw Mar-A-Lago Fundraiser For Woman A...,0


In [ ]:
ff.tail()

,text,label
45752,Heavily-Armed Man Claiming To Be Jesus Arrest...,0
45753,Evangelicals Across The Spectrum Are Clarifyin...,0
45754,"Senators question Kaleo' $4,500 tag on opioid ...",1
45755,CNN Does NOT Hold Back; Completely Makes Fun ...,0
45756,U.S. judge throws out Texas voter ID law suppo...,1


In [ ]:
ff.shape

(45757, 2)

In [ ]:
ff.describe()


,label
count,45757.000000
mean,0.500470
std,0.500005
min,0.000000
25%,0.000000
50%,1.000000
75%,1.000000
max,1.000000


In [ ]:
ff.info()

<class 'pandas.DataFrame'>
RangeIndex: 45757 entries, 0 to 45756
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    45757 non-null  str  
 1   label   45757 non-null  int64
dtypes: int64(1), str(1)
memory usage: 715.1 KB


In [ ]:
ff.isnull().sum().reset_index()

,index,0
0,text,0
1,label,0


In [ ]:
ff[("label")].value_counts()

label
1    22900
0    22857
Name: count, dtype: int64

# 🎯 Separate Features and Labels

The **text** column is used as the input feature (X), and the **label** column is used as the target variable (y).

In [ ]:
X = ff["text"]
y = ff["label"]

print(X.shape)
print(y.shape)

(45757,)
(45757,)


# ✂️ Split the Dataset

Split the dataset into training and testing sets.

- Training Data: 80%
- Testing Data: 20%

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape)
print(X_test.shape)

(36605,)
(9152,)


# 🔢 Convert Text into Numerical Features

Machine learning models cannot understand raw text.

TF-IDF (Term Frequency-Inverse Document Frequency) converts each news article into a numerical feature vector based on the importance of words.

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)
print(X_train.shape)
print(X_test.shape)

(36605, 5000)
(9152, 5000)


# 🔄 Convert Data into PyTorch Tensors

Convert the processed NumPy arrays into PyTorch tensors so they can be used for training the neural network.

In [ ]:
X_train_tensor = torch.from_numpy(X_train.toarray()).float()
X_test_tensor = torch.from_numpy(X_test.toarray()).float()

y_train_tensor = torch.from_numpy(y_train.to_numpy()).long()
y_test_tensor = torch.from_numpy(y_test.to_numpy()).long()

C:\Users\Yadu_03\AppData\Local\Temp\ipykernel_8208\3093884761.py:4: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  y_train_tensor = torch.from_numpy(y_train.to_numpy()).long()


In [ ]:
print(X_train_tensor.shape)
print(X_test_tensor.shape)
print(y_train_tensor.shape)
print(y_test_tensor.shape)

torch.Size([36605, 5000])
torch.Size([9152, 5000])
torch.Size([36605])
torch.Size([9152])


# 📦 Create TensorDataset and DataLoader

TensorDataset combines input features and labels.

DataLoader loads the dataset in batches, making training more efficient and memory-friendly.

In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
for X_batch, y_batch in train_loader:
    print(X_batch.shape)
    print(y_batch.shape)
    break

torch.Size([64, 5000])
torch.Size([64])


# 🧠 Build the Neural Network

Create a Feedforward Neural Network consisting of:

- Input Layer
- Hidden Layer 1
- Hidden Layer 2
- Output Layer

ReLU activation is used for introducing non-linearity, and Dropout helps reduce overfitting.

In [ ]:
class FakeNewsClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(5000, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 2)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):

        x = self.relu(self.fc1(x))
        x = self.dropout(x)

        x = self.relu(self.fc2(x))
        x = self.dropout(x)

        x = self.fc3(x)

        return x

In [ ]:
model = FakeNewsClassifier()
print(model)

FakeNewsClassifier(
  (fc1): Linear(in_features=5000, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=2, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
)


# ⚙️ Define Loss Function and Optimizer

- Loss Function: CrossEntropyLoss
- Optimizer: Adam

These are used to measure prediction error and update the model's weights during training.

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

# 🚀 Train the Model

Train the neural network over multiple epochs.

For each epoch:

1. Perform a forward pass.
2. Calculate the loss.
3. Perform backpropagation.
4. Update the model parameters.

In [ ]:
epochs = 20

for epoch in range(epochs):

    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

Epoch [1/20], Loss: 0.1158
Epoch [2/20], Loss: 0.0348
Epoch [3/20], Loss: 0.0176
Epoch [4/20], Loss: 0.0087
Epoch [5/20], Loss: 0.0049
Epoch [6/20], Loss: 0.0041
Epoch [7/20], Loss: 0.0051
Epoch [8/20], Loss: 0.0026
Epoch [9/20], Loss: 0.0031
Epoch [10/20], Loss: 0.0021
Epoch [11/20], Loss: 0.0013
Epoch [12/20], Loss: 0.0024
Epoch [13/20], Loss: 0.0012
Epoch [14/20], Loss: 0.0013
Epoch [15/20], Loss: 0.0012


# 📊 Model Evaluation

After training the neural network, evaluate its performance on the test dataset.

The following evaluation metrics are used:

- **Accuracy** – Measures the overall percentage of correctly classified news articles.
- **Precision** – Measures how many predicted positive instances are actually correct.
- **Recall** – Measures how many actual positive instances are correctly identified.
- **F1-Score** – The harmonic mean of precision and recall.
- **Confusion Matrix** – Summarizes the number of correct and incorrect predictions for each class.

These metrics provide a comprehensive understanding of the model's classification performance.

In [ ]:
model.eval()

predictions = []
actuals = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        outputs = model(X_batch)

        _, predicted = torch.max(outputs, 1)

        predictions.extend(predicted.numpy())
        actuals.extend(y_batch.numpy())

## ✅ Calculate Accuracy

Accuracy represents the proportion of correctly classified news articles out of the total number of articles in the test dataset.

In [ ]:
accuracy = accuracy_score(actuals, predictions)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9783


## 📋 Classification Report

The classification report summarizes the model's performance using:

- Precision
- Recall
- F1-Score
- Support

These metrics are calculated separately for each class.

In [ ]:
print(classification_report(actuals, predictions))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98      4572
           1       0.98      0.98      0.98      4580

    accuracy                           0.98      9152
   macro avg       0.98      0.98      0.98      9152
weighted avg       0.98      0.98      0.98      9152



## 📌 Confusion Matrix

The confusion matrix compares the model's predicted labels with the actual labels.

It provides:

- **True Positives (TP)**
- **True Negatives (TN)**
- **False Positives (FP)**
- **False Negatives (FN)**

This helps identify the types of classification errors made by the model.

In [ ]:
print(confusion_matrix(actuals, predictions))

[[4475   97]
 [ 102 4478]]


# 📰 Predict on New News

Use the trained model to classify new news articles as either:

- Fake News
- Real News

The input text is first converted using the trained TF-IDF vectorizer before making predictions.

In [ ]:
news = ["The government announced a new education policy."]

news_vector = vectorizer.transform(news)
news_tensor = torch.from_numpy(news_vector.toarray()).float()

model.eval()

with torch.no_grad():
    output = model(news_tensor)
    prediction = torch.argmax(output, dim=1).item()

if prediction == 0:
    print("Fake News")
else:
    print("Real News")

Fake News


# ✅ Conclusion

A Fake News Detection model was successfully developed using TF-IDF Vectorization and a Feedforward Neural Network in PyTorch.

The model achieved high classification performance with an accuracy of approximately 98%, demonstrating its effectiveness in distinguishing between fake and real news articles.

# 📌 Technologies Used

- Python
- Pandas
- Scikit-learn
- PyTorch
- NumPy
- Jupyter Notebook

# 💾 Save the Trained Model and TF-IDF Vectorizer

After successfully training the neural network, the model and the TF-IDF vectorizer are saved to disk.

Saving them allows the trained model to be reused later without retraining.

The following files are created:

- **fake_news_model.pt** – Stores the trained neural network weights.
- **tfidf_vectorizer.pkl** – Stores the trained TF-IDF vocabulary and feature mapping.

These files are required when deploying the model in a Streamlit web application for making predictions on new news articles.

In [ ]:
import joblib
torch.save(model.state_dict(), "fake_news_model.pt")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']